# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets and their fields. For precise referencing, all entities are referenced by their `@id` fields.

Let's print the available record sets, their IDs, and their main fields.

In [ ]:
# Print available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"There are {len(record_sets)} record sets available.")

for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (ID: {rs.id})")
    # Each record set can have multiple fields
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields (by @id):")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, dataType: {getattr(field, 'data_type', 'N/A')})")
    else:
        print("  (No fields found)")

## Examine Record Preview
Let's view a preview of actual records for one of the record sets using its `@id`.

In [ ]:
# Replace this with the actual @id you'd like to inspect. We'll pick the first available record set for the demo.
if record_sets:
    first_record_set = record_sets[0]
    print(f"\nPreview of records for record set '{first_record_set.name}' (@id: {first_record_set.id}):")
    for i, record in enumerate(dataset.records(record_set=first_record_set.id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets available in this dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities use their `@id` fields as references.

We'll extract all tabular record sets into Pandas DataFrames and preview the fields and first rows for one of them.

In [ ]:
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nDataFrame for record set @id '{record_set_id}': columns = {df.columns.tolist()}")
        print(df.head(2))
    else:
        print(f"\nRecord set @id '{record_set_id}' yields no records.")

## 4. Exploratory Data Analysis (EDA)
Apply exploratory data processing, such as filtering, normalization, outlier removal, and grouping. Operations reference each field by its `@id` in code.

Let's: 
- Pick a numeric field (by `@id`) and filter the DataFrame,
- Normalize that field,
- Optionally, group by a categorical field if available.

*Below, replace the values in angle brackets by valid field `@id` strings from the previous overview step, e.g., `'accession'`, `'coverage'`, `...` etc.*

In [ ]:
# -- Pick one record set and relevant fields --
# Example record set @id and field @id; substitute with actual ones printed above.
# For illustration, we'll select the first with numeric data if possible.

chosen_record_set_id = None
numeric_field_id = None
group_field_id = None

# Try to auto-detect a numeric field for demonstration
import numpy as np

for rs in record_sets:
    df = dataframes.get(rs.id)
    if df is not None:
        # Heuristically pick first numeric column
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_cols:
            chosen_record_set_id = rs.id
            numeric_field_id = numeric_cols[0]
            # Try to select another field for grouping that's not numeric
            non_numeric_cols = [c for c in df.columns if c != numeric_field_id]
            if non_numeric_cols:
                group_field_id = non_numeric_cols[0]
            break

if chosen_record_set_id is not None:
    print(f"Operating on record set @id: {chosen_record_set_id}")
    print(f"Numeric field for filtering and normalization: {numeric_field_id}")
    if group_field_id is not None:
        print(f"Field for optional grouping: {group_field_id}")
    df = dataframes[chosen_record_set_id]

    # Filter records, e.g., where numeric field > threshold
    threshold = df[numeric_field_id].quantile(0.75)  # Example: top 25% values
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where '{numeric_field_id}' > {threshold} (top quartile):")
    print(filtered_df.head())

    # Normalize the numeric field for filtered data
    normed = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    filtered_df[numeric_field_id + '_normalized'] = normed
    print(f"\nNormalized '{numeric_field_id}':")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Optionally, group by another field
    if group_field_id is not None:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        print(grouped.head())
else:
    print("No suitable record set and numeric field found for EDA.")

## 5. Visualization
Visualize data distributions and relationships between fields. 
All field references are noted by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if chosen_record_set_id is not None and numeric_field_id is not None:
    df = dataframes[chosen_record_set_id]
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in '{chosen_record_set_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group field is set, plot the mean for each group
    if group_field_id is not None:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped, x=group_field_id, y=numeric_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}'")
        plt.show()
else:
    print("Visualization skipped: No numeric field or record set selected.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. 

- We accessed metadata, record sets, and described every entity via its Croissant `@id`.
- Sample records were loaded and analyzed, including filtering and normalization of a numeric field, and grouping by a key attribute.
- Distribution and aggregated visualizations provided insight into the data.

**Next steps:** You may now design further feature engineering, build models, or augment this workflow using other columns and record sets as required.
